# Lab 06 – Landmark Recognition Using the SIFT Algorithm

**Course:** Introduction to Computer Vision – Lab
**Goal:** recognise famous landmarks in tourist photos by matching **SIFT** features against a database of reference images, reject landmarks that are not in the database, and localise the recognised landmark in the query image.

| Section | Assignment task |
|---|---|
| 0 | Setup – load the image dataset into Colab |
| 1 | Task 1 – Landmark image database |
| 2 | Task 2 – SIFT feature extraction (reference images) |
| 3 | Task 3 – Query image processing |
| 4 | Task 4 – Feature matching & landmark recognition |
| 5 | Task 5 – Landmark localisation (RANSAC homography) |
| 6 | Complete recognition pipeline (all 7 expected outputs) |
| 7 | Task 6 – Performance evaluation |
| 8 | Discussion – performance and limitations |

**Pipeline:** query photo → grayscale (+ contrast normalisation) → SIFT keypoints & 128-D descriptors → brute-force k-NN matching (k = 2, Euclidean) against every reference image → Lowe's ratio test → rank landmarks → RANSAC homography (geometric verification) → accept / reject → draw the landmark region.

> **How to run in Google Colab:** `Runtime → Run all`. The first cell asks you to upload **`landmark_dataset.zip`** (see Section 0).

## 0. Setup – load the dataset

The dataset is supplied as **`landmark_dataset.zip`**:

```
landmark_dataset.zip
├── landmark_database/            ← Task 1: reference images, one folder per landmark
│   ├── Colosseum/  Eiffel_Tower/  Great_Wall_of_China/  Statue_of_Liberty/  Taj_Mahal/
├── query_images/                 ← test photos that are NOT in the database
│   ├── <Landmark>/original.jpg   ← a different photo of the landmark (typical tourist view)
│   ├── <Landmark>/viewpoint.jpg  ← a photo from a different viewpoint / distance
│   ├── Unknown/                  ← landmarks that are not in the database
│   └── Challenging/              ← extra hard cases (optional stress test)
└── image_sources.csv             ← Wikimedia Commons source page of every image
```

**Option A – upload (default):** run the cell below and click **Choose Files** → select `landmark_dataset.zip`. You can also drag the zip into the Colab *Files* panel (folder icon on the left) before running the cell; it will be found automatically.

**Option B – Google Drive:** upload the zip (or the extracted folders) to a Drive folder, set `DRIVE_FOLDER` below (e.g. `"/content/drive/MyDrive/CV_Lab06"`) and run the cell.

In [ ]:
import os
import zipfile
from pathlib import Path

DRIVE_FOLDER = None   # Option B, e.g. "/content/drive/MyDrive/CV_Lab06"


def dataset_root():
    """Return the folder that contains landmark_database/ and query_images/ (searches 2 levels deep)."""
    for pattern in ("", "*/", "*/*/"):
        for hit in Path(".").glob(pattern + "landmark_database"):
            if hit.is_dir() and (hit.parent / "query_images").is_dir():
                return hit.parent.resolve()
    return None


if DRIVE_FOLDER:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(DRIVE_FOLDER)

if dataset_root() is None:
    zips = sorted(Path(".").glob("*.zip"))
    if not zips:
        try:
            from google.colab import files
            print("Select landmark_dataset.zip ...")
            zips = [Path(name) for name in files.upload()]
        except ImportError:
            raise RuntimeError("Not in Colab: put landmark_database/ and query_images/ next to this notebook.")
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(".")
        print("Extracted", z.name)

root = dataset_root()
assert root is not None, "landmark_database/ and query_images/ not found - check the uploaded zip."
os.chdir(root)
print("Working directory:", Path.cwd())

In [ ]:
import time
import pickle
import textwrap

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display

print("OpenCV version:", cv2.__version__)
assert hasattr(cv2, "SIFT_create"), "This OpenCV build has no SIFT (OpenCV >= 4.4 is required)."

### Parameters

All thresholds of the system are defined in one place. They were chosen so that a decision is **never based on the largest match count alone** (Task 4): a landmark is accepted only if it has enough reliable matches, clearly beats the runner-up landmark, *and* passes RANSAC geometric verification.

In [ ]:
# ---------------- Folders ----------------
BASE_DIR  = Path.cwd()
DB_DIR    = BASE_DIR / "landmark_database"
QUERY_DIR = BASE_DIR / "query_images"
OUT_DIR   = BASE_DIR / "outputs"
SHOT_DIR  = OUT_DIR / "screenshots"           # every result figure is saved here
GEN_DIR   = OUT_DIR / "generated_queries"     # rotated / resized / darkened test images
for d in (OUT_DIR, SHOT_DIR, GEN_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------------- Parameters ----------------
MAX_DIM          = 800    # images are downscaled so the longest side is at most 800 px
USE_CLAHE        = True   # contrast normalisation of the grayscale image before SIFT
RATIO_TEST       = 0.75   # Lowe's ratio test: best distance < 0.75 * second-best distance
MIN_GOOD_MATCHES = 20     # min. reliable matches for the best landmark (also needed to fit a homography)
MIN_MARGIN       = 1.5    # best landmark must have >= 1.5x the reliable matches of the runner-up
MIN_INLIERS      = 15     # min. RANSAC inliers (geometric verification)
RANSAC_THRESH    = 5.0    # RANSAC reprojection threshold in pixels

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEQ_BLUE = LinearSegmentedColormap.from_list(          # single-hue sequential scale for heatmaps
    "seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
OK_COLOR, BAD_COLOR = "#1a7f37", "#c62828"
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})


def pretty(label):
    return label.replace("_", " ")


def to_rgb(img):
    return img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def list_images(folder):
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_EXTS)


def load_image(path, max_dim=MAX_DIM):
    """Read an image (BGR) and downscale it so that its longest side is at most max_dim."""
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    h, w = img.shape[:2]
    scale = max_dim / max(h, w)
    if scale < 1:
        img = cv2.resize(img, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
    return img


def save_figure(fig, name):
    fig.savefig(SHOT_DIR / f"{name}.png", dpi=110, bbox_inches="tight")

## 1. Task 1 – Landmark image database

The database contains **five famous landmarks** with **five reference images each**, taken from different viewpoints, distances and lighting conditions (day, sunset, night). Each landmark has its own folder named after the landmark – the folder name is the class label. All images come from Wikimedia Commons (sources in `image_sources.csv`).

The query photos in `query_images/` are **different photographs** that are not part of the database, so the evaluation in Task 6 is not just matching an image against itself.

In [ ]:
landmark_paths = {d.name: list_images(d) for d in sorted(DB_DIR.iterdir()) if d.is_dir()}
LANDMARKS = list(landmark_paths)

db_summary = pd.DataFrame({
    "Landmark": [pretty(lm) for lm in LANDMARKS],
    "Folder": [f"landmark_database/{lm}/" for lm in LANDMARKS],
    "Reference images": [len(landmark_paths[lm]) for lm in LANDMARKS],
    "Files": [", ".join(p.name for p in landmark_paths[lm]) for lm in LANDMARKS],
})
display(db_summary)
print("Total reference images:", db_summary["Reference images"].sum())

In [ ]:
n_cols = max(len(p) for p in landmark_paths.values())
fig, axes = plt.subplots(len(LANDMARKS), n_cols, figsize=(3.3 * n_cols, 2.7 * len(LANDMARKS)), squeeze=False)
for r, lm in enumerate(LANDMARKS):
    for c in range(n_cols):
        ax = axes[r, c]
        ax.axis("off")
        if c < len(landmark_paths[lm]):
            ax.imshow(to_rgb(load_image(landmark_paths[lm][c], 400)))
            ax.set_title(landmark_paths[lm][c].name, fontsize=8)
    axes[r, 0].text(-0.06, 0.5, pretty(lm), transform=axes[r, 0].transAxes, rotation=90,
                    ha="right", va="center", fontsize=11, fontweight="bold")
fig.suptitle("Task 1 – Landmark reference database (one row per landmark folder)", fontsize=13)
plt.tight_layout()
save_figure(fig, "task1_database")
plt.show()

## 2. Task 2 – SIFT feature extraction

SIFT (Scale-Invariant Feature Transform, Lowe 2004) works in four stages:

1. **Scale-space extrema detection** – the image is blurred with Gaussians of increasing σ; adjacent blurred images are subtracted (Difference of Gaussians, DoG). A pixel is a candidate if it is larger/smaller than its 26 neighbours (8 in its own DoG level + 9 above + 9 below).
2. **Keypoint localisation** – low-contrast candidates and edge responses are removed.
3. **Orientation assignment** – a 36-bin gradient-orientation histogram gives each keypoint a dominant direction → **rotation invariance**.
4. **Descriptor** – a 16×16 window around the keypoint (rotated to its orientation, at its scale) is split into 4×4 cells, each with an 8-bin orientation histogram → **4·4·8 = 128-D vector**, normalised → robust to illumination changes.

**Steps implemented below:** (1) load each reference image and convert it to grayscale, (2) detect keypoints, (3) compute descriptors, (4) display the keypoints, (5) store the descriptors with their landmark labels.

**Contrast normalisation (CLAHE).** SIFT's descriptors cope with brightness changes, but its *detector* uses a fixed contrast threshold, so a dark, low-contrast photo yields far fewer keypoints. Applying CLAHE (Contrast-Limited Adaptive Histogram Equalisation) to the grayscale image – identically for reference and query images – restores the keypoints. Section 7 shows the effect with and without it.

In [ ]:
sift = cv2.SIFT_create()
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def to_gray(img_bgr):
    """Step 1: grayscale conversion (+ optional CLAHE contrast normalisation)."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return clahe.apply(gray) if USE_CLAHE else gray


def extract_sift(img_bgr):
    """Steps 1-3: grayscale -> SIFT keypoints -> 128-D descriptors."""
    gray = to_gray(img_bgr)
    keypoints, descriptors = sift.detectAndCompute(gray, None)
    return gray, keypoints, descriptors


def draw_keypoints(img_bgr, keypoints, max_kp=300):
    """Step 4: draw the strongest keypoints; circle size = scale, radius line = orientation."""
    strongest = sorted(keypoints, key=lambda k: k.response, reverse=True)[:max_kp]
    return cv2.drawKeypoints(img_bgr, strongest, None, color=(0, 255, 0),
                             flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)


def build_database(paths_by_landmark):
    """Extract SIFT features of every reference image and keep them with their landmark label."""
    db = []
    for label, paths in paths_by_landmark.items():
        for path in paths:
            img = load_image(path)
            gray, kp, des = extract_sift(img)
            db.append({"label": label, "file": path.name, "image": img, "gray": gray,
                       "keypoints": kp, "descriptors": des})
    return db

In [ ]:
t0 = time.perf_counter()
database = build_database(landmark_paths)
print(f"SIFT features extracted from {len(database)} reference images in {time.perf_counter() - t0:.2f} s")

feature_table = pd.DataFrame([{
    "Landmark": pretty(r["label"]),
    "Image": r["file"],
    "Size (W x H)": f'{r["image"].shape[1]} x {r["image"].shape[0]}',
    "Keypoints": len(r["keypoints"]),
    "Descriptor matrix": str(None if r["descriptors"] is None else r["descriptors"].shape),
} for r in database])
display(feature_table)

### Step by step on one reference image

Original → grayscale (+ CLAHE) → SIFT keypoints → one keypoint's image patch and its 128-D descriptor (16 spatial cells × 8 orientation bins).

In [ ]:
example = database[0]
idx = int(np.argmax([k.response for k in example["keypoints"]]))   # strongest keypoint
kp = example["keypoints"][idx]
x, y, r = int(kp.pt[0]), int(kp.pt[1]), int(max(8, 1.5 * kp.size))
gray = example["gray"]
patch = gray[max(0, y - r):y + r, max(0, x - r):x + r]

fig, axes = plt.subplots(1, 5, figsize=(22, 4.5), gridspec_kw={"width_ratios": [3, 3, 3, 1.4, 1.6]})
axes[0].imshow(to_rgb(example["image"]));  axes[0].set_title(f'Reference image: {example["file"]}')
axes[1].imshow(gray, cmap="gray");          axes[1].set_title("Grayscale" + (" + CLAHE" if USE_CLAHE else ""))
axes[2].imshow(to_rgb(draw_keypoints(example["image"], example["keypoints"])))
axes[2].set_title(f'{len(example["keypoints"])} SIFT keypoints (300 strongest drawn)')
axes[3].imshow(patch, cmap="gray");         axes[3].set_title(f"Patch of strongest keypoint\nscale={kp.size:.1f}, angle={kp.angle:.0f}°")
for ax in axes[:4]:
    ax.axis("off")
im = axes[4].imshow(example["descriptors"][idx].reshape(16, 8), cmap=SEQ_BLUE, aspect="auto")
axes[4].set_title("128-D descriptor")
axes[4].set_xlabel("orientation bin (8 x 45°)")
axes[4].set_ylabel("spatial cell (4 x 4 grid)")
axes[4].tick_params(length=0)
for s in axes[4].spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=axes[4], fraction=0.08)
plt.tight_layout()
save_figure(fig, "task2_sift_steps")
plt.show()

### Reference images with their detected keypoints

In [ ]:
fig, axes = plt.subplots(len(LANDMARKS), n_cols, figsize=(3.3 * n_cols, 2.7 * len(LANDMARKS)), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for r, lm in enumerate(LANDMARKS):
    refs = [d for d in database if d["label"] == lm]
    for c, ref in enumerate(refs):
        axes[r, c].imshow(to_rgb(draw_keypoints(ref["image"], ref["keypoints"], max_kp=200)))
        axes[r, c].set_title(f'{ref["file"]}: {len(ref["keypoints"])} keypoints', fontsize=8)
    axes[r, 0].text(-0.06, 0.5, pretty(lm), transform=axes[r, 0].transAxes, rotation=90,
                    ha="right", va="center", fontsize=11, fontweight="bold")
fig.suptitle("Task 2 – SIFT keypoints of the reference images (200 strongest drawn per image)", fontsize=13)
plt.tight_layout()
save_figure(fig, "task2_reference_keypoints")
plt.show()

### Step 5 – store the descriptors with their landmark labels

`cv2.KeyPoint` objects cannot be pickled, so each keypoint is stored as `(x, y, size, angle, response, octave, class_id)` next to its descriptor matrix and the landmark label. The file is reloaded to check that nothing was lost – a deployed system would load this file instead of re-extracting features at start-up.

In [ ]:
def serialize_keypoints(keypoints):
    return np.array([(k.pt[0], k.pt[1], k.size, k.angle, k.response, k.octave, k.class_id)
                     for k in keypoints], dtype=np.float64)


def deserialize_keypoints(array):
    return [cv2.KeyPoint(float(x), float(y), float(s), float(a), float(r), int(o), int(c))
            for x, y, s, a, r, o, c in array]


feature_db = [{"label": r["label"], "file": r["file"], "image_shape": r["image"].shape[:2],
               "keypoints": serialize_keypoints(r["keypoints"]), "descriptors": r["descriptors"]}
              for r in database]
FEATURE_FILE = OUT_DIR / "sift_feature_database.pkl"
with open(FEATURE_FILE, "wb") as f:
    pickle.dump(feature_db, f)

with open(FEATURE_FILE, "rb") as f:
    reloaded = pickle.load(f)
restored = deserialize_keypoints(reloaded[0]["keypoints"])
print(f"Saved {len(reloaded)} labelled entries to {FEATURE_FILE.name} ({FEATURE_FILE.stat().st_size / 1e6:.1f} MB)")
print("Round-trip check - keypoints:", len(restored) == len(database[0]["keypoints"]),
      "| descriptors:", np.array_equal(reloaded[0]["descriptors"], database[0]["descriptors"]))

display(pd.DataFrame([{"Landmark": pretty(e["label"]), "Descriptors": len(e["descriptors"])} for e in reloaded])
        .groupby("Landmark").agg(Images=("Descriptors", "size"), Total_descriptors=("Descriptors", "sum")))

## 3. Task 3 – Query image processing

`process_query()` accepts an unknown landmark photo – a **file path** or an image array – and (1) loads it, (2) converts it to grayscale, (3) detects SIFT keypoints, (4) extracts descriptors and (5) displays the keypoints. The query goes through exactly the same preprocessing as the reference images.

To test your own photo: upload it with the Colab *Files* panel and set `QUERY_PATH` to its name, e.g. `QUERY_PATH = Path("my_photo.jpg")`.

In [ ]:
def process_query(query, show=True, title="Query image"):
    """Load (path) or accept (BGR array) a query image and extract its SIFT features."""
    img = load_image(query) if isinstance(query, (str, Path)) else query
    gray, kp, des = extract_sift(img)
    if show:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
        panels = [(img, title), (gray, "Grayscale" + (" + CLAHE" if USE_CLAHE else "")),
                  (draw_keypoints(img, kp), f"SIFT keypoints: {len(kp)} (300 strongest drawn)")]
        for ax, (im, t) in zip(axes, panels):
            ax.imshow(to_rgb(im), cmap="gray")
            ax.set_title(t)
            ax.axis("off")
        plt.tight_layout()
        plt.show()
    return {"image": img, "gray": gray, "keypoints": kp, "descriptors": des}

In [ ]:
QUERY_PATH = QUERY_DIR / "Taj_Mahal" / "original.jpg"    # <- change to any image you want to recognise

query = process_query(QUERY_PATH, title=f"Query: {QUERY_PATH.name}")
print(f"Keypoints: {len(query['keypoints'])} | descriptor matrix: {query['descriptors'].shape}")

## 4. Task 4 – Feature matching and landmark recognition

1. **Brute-Force matcher, Euclidean (L2) distance** – SIFT descriptors are real-valued vectors, so `cv2.NORM_L2` is the correct metric.
2. **k-NN matching with k = 2** – for each query descriptor, find its two nearest reference descriptors.
3. **Lowe's ratio test** – keep a match only if `d1 < 0.75 · d2`. If the second-best match is almost as close, the match is ambiguous (e.g. repeated windows or bricks) and is discarded. Additionally each reference keypoint may be used **only once** (keep its closest match); otherwise many query points can pile onto one reference point and fool RANSAC later.
4. **Count reliable matches per landmark** – the query is matched against every reference image separately; a landmark's score is its best-matching reference image.
5. **Rank** the landmarks by their number of reliable matches.
6. **Decision** – the top landmark is accepted only if it passes **all** checks:

| Check | Threshold | Why |
|---|---|---|
| Reliable matches | ≥ `MIN_GOOD_MATCHES` = 20 | enough evidence at all |
| Margin over runner-up | ≥ `MIN_MARGIN` = 1.5× | the winner must be *clearly* better than the 2nd landmark – unknown landmarks give similar small counts for every landmark |
| RANSAC inliers (Task 5) | ≥ `MIN_INLIERS` = 15 | the matches must agree on one geometric transformation – random matches do not |

The largest match count alone is not enough: a photo of a landmark that is *not* in the database still produces 20–30 accidental matches with some reference image, but those matches are spread evenly over all landmarks and are geometrically inconsistent.

In [ ]:
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)    # Euclidean distance between 128-D descriptors


def reliable_matches(des_query, des_ref, ratio=RATIO_TEST):
    """k-NN matching (k=2) + Lowe's ratio test; every reference keypoint is used at most once."""
    if des_query is None or des_ref is None or len(des_query) < 2 or len(des_ref) < 2:
        return []
    best = {}
    for pair in bf.knnMatch(des_query, des_ref, k=2):
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < ratio * n.distance:
            if m.trainIdx not in best or m.distance < best[m.trainIdx].distance:
                best[m.trainIdx] = m
    return list(best.values())


def rank_landmarks(query):
    """Match the query with every reference image and keep the best reference per landmark."""
    best = {}
    for i, ref in enumerate(database):
        good = reliable_matches(query["descriptors"], ref["descriptors"])
        lm = ref["label"]
        if lm not in best or len(good) > len(best[lm]["matches"]):
            best[lm] = {"landmark": lm, "ref_index": i, "matches": good}
    return sorted(best.values(), key=lambda c: len(c["matches"]), reverse=True)


def ranking_table(candidates):
    return pd.DataFrame({
        "Rank": range(1, len(candidates) + 1),
        "Landmark": [pretty(c["landmark"]) for c in candidates],
        "Best reference image": [database[c["ref_index"]]["file"] for c in candidates],
        "Reliable matches": [len(c["matches"]) for c in candidates],
    }).set_index("Rank")


def matching_checks(candidates):
    """Match-count checks of Task 4 (the RANSAC check is added in Task 5)."""
    n_best, n_second = len(candidates[0]["matches"]), len(candidates[1]["matches"])
    margin = n_best / max(n_second, 1)
    return [(f"Reliable matches >= {MIN_GOOD_MATCHES}", n_best >= MIN_GOOD_MATCHES, n_best),
            (f"Margin over runner-up >= {MIN_MARGIN}x", margin >= MIN_MARGIN, round(margin, 2))]


def draw_match_lines(ref, query, matches, mask=None, max_lines=80):
    """Reference image on the left, query image on the right; the closest matches are drawn."""
    pairs = [cv2.DMatch(m.trainIdx, m.queryIdx, m.distance) for m in matches]   # ref -> query
    if mask is not None:
        pairs = [m for m, keep in zip(pairs, mask) if keep]
    pairs = sorted(pairs, key=lambda m: m.distance)[:max_lines]
    return cv2.drawMatches(ref["image"], ref["keypoints"], query["image"], query["keypoints"], pairs, None,
                           matchColor=(0, 255, 0), singlePointColor=(255, 0, 0),
                           flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

In [ ]:
candidates = rank_landmarks(query)
display(ranking_table(candidates))

best = candidates[0]
print(f"Most likely landmark: {pretty(best['landmark'])} with {len(best['matches'])} reliable matches")
for name, ok, value in matching_checks(candidates):
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}  (value: {value})")

fig = plt.figure(figsize=(17, 6))
plt.imshow(to_rgb(draw_match_lines(database[best["ref_index"]], query, best["matches"])))
plt.title(f"Reliable matches: reference '{database[best['ref_index']]['file']}' (left) <-> query (right) "
          f"- {min(80, len(best['matches']))} closest of {len(best['matches'])} drawn")
plt.axis("off")
save_figure(fig, "task4_matches")
plt.show()

## 5. Task 5 – Landmark localisation

1. The reliable matches give pairs of corresponding points (reference ↔ query).
2. If there are at least `MIN_GOOD_MATCHES` of them, a **homography** `H` (3×3 perspective transformation) is estimated with **RANSAC**: random 4-point samples are tried and the `H` agreeing with the most matches (reprojection error < 5 px) wins. Those matches are the **inliers**.
3. The four corners of the reference image are projected into the query image with `H`.
4. The projected quadrilateral is drawn (red) together with the inlier keypoints (yellow). The canvas is padded with grey when the reference frame extends beyond the query photo.
5. The number of reliable matches and RANSAC inliers is reported.

**Validity checks** – no polygon is drawn (a message is shown instead) if: there are too few inliers; `det(H[:2,:2])` is ≤ 0.01 or ≥ 100 (the region collapses, flips or explodes); the projected outline is not convex (twisted); or it covers less than 2 % of the image.

*Note:* a homography is exact only for planar scenes. Landmarks are 3-D, but because they are photographed from far away it is a good approximation – which is also why very different viewpoints fail (Section 8).

In [ ]:
def check_homography(H, polygon, inliers, query_shape):
    """Reject transformations that would produce a meaningless polygon."""
    if inliers < MIN_INLIERS:
        return False, f"Only {inliers} RANSAC inliers (< {MIN_INLIERS}): the matches do not agree on one transformation."
    det = np.linalg.det(H[:2, :2])
    if not 0.01 < det < 100:
        return False, f"Degenerate transformation (det = {det:.3g}): the region would collapse, flip or explode."
    if not cv2.isContourConvex(polygon.astype(np.float32)):
        return False, "The projected outline is not convex (twisted / self-intersecting)."
    area_ratio = cv2.contourArea(polygon.astype(np.float32)) / (query_shape[0] * query_shape[1])
    if area_ratio < 0.02:
        return False, f"The projected outline is implausibly small ({area_ratio:.1%} of the image)."
    return True, "Homography is geometrically valid."


def localize_landmark(ref, query, matches):
    """RANSAC homography reference -> query and the projected outline of the reference image."""
    loc = {"H": None, "inlier_mask": None, "inliers": 0, "inlier_points": None,
           "polygon": None, "valid": False, "message": ""}
    if len(matches) < MIN_GOOD_MATCHES:
        loc["message"] = f"Only {len(matches)} reliable matches (< {MIN_GOOD_MATCHES}): not enough to estimate a homography."
        return loc
    src = np.float32([ref["keypoints"][m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([query["keypoints"][m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, RANSAC_THRESH)
    if H is None:
        loc["message"] = "RANSAC could not find a consistent homography."
        return loc
    mask = mask.ravel().astype(bool)
    h, w = ref["image"].shape[:2]
    corners = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]]).reshape(-1, 1, 2)
    polygon = cv2.perspectiveTransform(corners, H)
    loc.update(H=H, inlier_mask=mask, inliers=int(mask.sum()),
               inlier_points=dst.reshape(-1, 2)[mask], polygon=polygon)
    loc["valid"], loc["message"] = check_homography(H, polygon, loc["inliers"], query["image"].shape)
    return loc


def draw_localization(query, loc, color=(0, 0, 255)):
    """Draw the projected reference outline (red) and inlier keypoints (yellow) on the query image."""
    img = query["image"]
    if not loc["valid"]:
        return img.copy()
    h, w = img.shape[:2]
    pts = loc["polygon"].reshape(-1, 2)

    def pad(overshoot, limit):   # grey border so outline parts outside the photo stay visible
        return int(np.clip(np.ceil(overshoot) + 12, 0, limit)) if overshoot > 0 else 0

    left, top = pad(-pts[:, 0].min(), w), pad(-pts[:, 1].min(), h)
    right, bottom = pad(pts[:, 0].max() - w, w), pad(pts[:, 1].max() - h, h)
    canvas = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(215, 215, 215))
    thick = max(2, max(h, w) // 250)
    for px, py in loc["inlier_points"]:
        cv2.circle(canvas, (int(px) + left, int(py) + top), thick + 1, (0, 255, 255), -1, cv2.LINE_AA)
    shifted = np.int32(pts + [left, top])
    cv2.polylines(canvas, [shifted.reshape(-1, 1, 2)], True, color, thick + 1, cv2.LINE_AA)
    for px, py in shifted:
        cv2.circle(canvas, (int(px), int(py)), 3 * thick, color, -1, cv2.LINE_AA)
    return canvas

In [ ]:
best = candidates[0]
ref = database[best["ref_index"]]
loc = localize_landmark(ref, query, best["matches"])
print(f"Landmark: {pretty(best['landmark'])} | reliable matches: {len(best['matches'])} | RANSAC inliers: {loc['inliers']}")
print("Geometry:", loc["message"])

fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={"width_ratios": [2, 1]})
axes[0].imshow(to_rgb(draw_match_lines(ref, query, best["matches"], mask=loc["inlier_mask"])))
axes[0].set_title(f"RANSAC inliers: {loc['inliers']} of {len(best['matches'])} reliable matches (up to 80 drawn)")
if loc["valid"]:
    axes[1].imshow(to_rgb(draw_localization(query, loc)))
    axes[1].set_title("Localised landmark: projected reference outline")
else:
    axes[1].imshow(to_rgb(query["image"]))
    axes[1].set_title("No polygon drawn:\n" + textwrap.fill(loc["message"], 50), color=BAD_COLOR)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
save_figure(fig, "task5_localization")
plt.show()

**What happens with an unreliable transformation?** Below, the same query is deliberately localised against the *runner-up* landmark's best reference image. The few accidental matches do not agree on one transformation, so the system prints a message instead of drawing an invalid polygon.

In [ ]:
wrong = candidates[1]
loc_wrong = localize_landmark(database[wrong["ref_index"]], query, wrong["matches"])
print(f"Query vs {pretty(wrong['landmark'])}: {len(wrong['matches'])} reliable matches, {loc_wrong['inliers']} RANSAC inliers")
print("Polygon drawn:", loc_wrong["valid"], "->", loc_wrong["message"])

## 6. Complete recognition pipeline

`recognize_landmark()` chains Tasks 3–5 and produces all **expected outputs**:

1. the query image, 2. its SIFT keypoints, 3. the matched features between the best reference image and the query, 4. the predicted landmark name, 5. the number of reliable matches (and RANSAC inliers), 6. the localised landmark region when geometric verification succeeds, 7. a message saying whether the landmark was **recognised** or **rejected**.

The processing time covers feature extraction, matching against the whole database and RANSAC (not the plotting). Every result figure is saved in `outputs/screenshots/`.

In [ ]:
def recognize_landmark(query_input, name="query", show=True, save=True):
    t0 = time.perf_counter()
    query = process_query(query_input, show=False)
    candidates = rank_landmarks(query)
    best, runner_up = candidates[0], candidates[1]
    ref = database[best["ref_index"]]
    loc = localize_landmark(ref, query, best["matches"])
    checks = matching_checks(candidates) + [
        (f"RANSAC inliers >= {MIN_INLIERS}", loc["inliers"] >= MIN_INLIERS, loc["inliers"])]
    recognized = all(ok for _, ok, _ in checks)
    elapsed = time.perf_counter() - t0

    result = {"name": name, "query": query, "candidates": candidates, "ref": ref, "loc": loc,
              "checks": checks, "recognized": recognized, "top_candidate": best["landmark"],
              "prediction": best["landmark"] if recognized else "Unknown",
              "n_keypoints": len(query["keypoints"]), "n_good": len(best["matches"]),
              "n_inliers": loc["inliers"], "runner_up": runner_up["landmark"],
              "margin": len(best["matches"]) / max(len(runner_up["matches"]), 1), "time": elapsed}
    if show or save:
        fig = plot_result(result)
        if save:
            save_figure(fig, name)
        if show:
            plt.show()
            display(ranking_table(candidates))
        else:
            plt.close(fig)
    return result


def plot_result(r):
    q, ref, loc = r["query"], r["ref"], r["loc"]
    best, runner_up = r["candidates"][0], r["candidates"][1]
    fig = plt.figure(figsize=(18, 10.5))
    gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.1])
    ax_q, ax_kp, ax_loc = (fig.add_subplot(gs[0, i]) for i in range(3))
    ax_m, ax_txt = fig.add_subplot(gs[1, :2]), fig.add_subplot(gs[1, 2])

    ax_q.imshow(to_rgb(q["image"]))
    ax_q.set_title("1. Query image")
    ax_kp.imshow(to_rgb(draw_keypoints(q["image"], q["keypoints"])))
    ax_kp.set_title(f"2. SIFT keypoints: {len(q['keypoints'])} (300 strongest drawn)")
    ax_m.imshow(to_rgb(draw_match_lines(ref, q, best["matches"], loc["inlier_mask"])))
    shown = "RANSAC inliers" if loc["inlier_mask"] is not None else "reliable matches"
    ax_m.set_title(f"3. Matched features: reference '{ref['file']}' (left) <-> query (right) - {shown} drawn")

    if r["recognized"] and loc["valid"]:
        ax_loc.imshow(to_rgb(draw_localization(q, loc)))
        ax_loc.set_title("6. Localised landmark region")
    else:
        why = "Landmark rejected." if not r["recognized"] else loc["message"]
        ax_loc.imshow(to_rgb(q["image"]))
        ax_loc.set_title("6. No region drawn: " + textwrap.fill(why, 45), fontsize=10, color=BAD_COLOR)

    status = "RECOGNIZED" if r["recognized"] else "REJECTED - landmark not in database"
    lines = [
        f"4. Predicted landmark : {pretty(r['prediction']) if r['recognized'] else 'Unknown'}",
        f"   Top candidate      : {pretty(r['top_candidate'])}",
        f"5. Reliable matches   : {r['n_good']}",
        f"   RANSAC inliers     : {r['n_inliers']}",
        f"   Runner-up          : {pretty(runner_up['landmark'])} ({len(runner_up['matches'])} matches)",
        f"   Processing time    : {r['time']:.2f} s",
        "",
        "Decision checks:",
        *[f"  [{'PASS' if ok else 'FAIL'}] {name} ({value})" for name, ok, value in r["checks"]],
        "",
        "Geometry: " + textwrap.fill(loc["message"] or "-", 40, subsequent_indent="          "),
    ]
    ax_txt.text(0.0, 0.97, "\n".join(lines), family="monospace", fontsize=10.5, va="top", transform=ax_txt.transAxes)
    ax_txt.text(0.0, 0.03, "7. " + status, fontsize=13, fontweight="bold", transform=ax_txt.transAxes,
                color=OK_COLOR if r["recognized"] else BAD_COLOR)
    for ax in (ax_q, ax_kp, ax_loc, ax_m, ax_txt):
        ax.axis("off")
    headline = f"RECOGNIZED: {pretty(r['prediction'])}" if r["recognized"] else "REJECTED: unknown landmark"
    fig.suptitle(f"{headline}   ({r['name']})", fontsize=16, fontweight="bold",
                 color=OK_COLOR if r["recognized"] else BAD_COLOR)
    fig.tight_layout()
    return fig

### Demo 1 – a landmark from the database (`QUERY_PATH`)

In [ ]:
result = recognize_landmark(QUERY_PATH, name="demo_" + QUERY_PATH.parent.name)

### Demo 2 – a landmark that is not in the database

In [ ]:
result_unknown = recognize_landmark(QUERY_DIR / "Unknown" / "big_ben.jpg", name="demo_unknown_big_ben")

## 7. Task 6 – Performance evaluation

For every landmark, the held-out photo `query_images/<Landmark>/original.jpg` is tested together with three synthetic variants of it, plus a second photo from a different viewpoint:

| Condition | How it is produced |
|---|---|
| Original | a different photo of the landmark (not in the database) |
| Rotated 45° | original rotated by 45°, canvas enlarged so nothing is cut off |
| Resized 50% | original scaled to half its width and height |
| Brightness 40% | every pixel multiplied by 0.4 (much darker, lower contrast) |
| Different viewpoint | `viewpoint.jpg`, another photo taken from a different position / distance |
| Not in database | 5 photos of other landmarks (Pyramids of Giza, Big Ben, Sydney Opera House, Leaning Tower of Pisa, Pisa cathedral square) |

For each test the predicted landmark, number of keypoints, number of reliable matches, RANSAC inliers, recognition decision and processing time are recorded. A test is **correct** if a known landmark is recognised with the right name, or if an unknown landmark is rejected.

In [ ]:
def rotate_image(img, angle=45):
    """Rotate about the centre and enlarge the canvas so that no part of the image is cut off."""
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    new_w, new_h = int(h * sin + w * cos), int(h * cos + w * sin)
    M[0, 2] += new_w / 2 - w / 2
    M[1, 2] += new_h / 2 - h / 2
    return cv2.warpAffine(img, M, (new_w, new_h))


def resize_image(img, factor=0.5):
    return cv2.resize(img, None, fx=factor, fy=factor, interpolation=cv2.INTER_AREA)


def darken_image(img, factor=0.4):
    return cv2.convertScaleAbs(img, alpha=factor, beta=0)


CONDITIONS = {"original": "Original", "rotated": "Rotated 45°", "resized": "Resized 50%",
              "dark": "Brightness 40%", "viewpoint": "Different viewpoint"}

test_cases = []   # (test id, condition, ground truth, image)
for lm in LANDMARKS:
    original = load_image(QUERY_DIR / lm / "original.jpg")
    variants = {"original": original, "rotated": rotate_image(original, 45),
                "resized": resize_image(original, 0.5), "dark": darken_image(original, 0.4),
                "viewpoint": load_image(QUERY_DIR / lm / "viewpoint.jpg")}
    for key, img in variants.items():
        test_cases.append((f"{lm}__{key}", CONDITIONS[key], lm, img))
for path in list_images(QUERY_DIR / "Unknown"):
    test_cases.append((f"Unknown__{path.stem}", "Not in database", "Unknown", load_image(path)))

for tid, _, _, img in test_cases:
    cv2.imwrite(str(GEN_DIR / f"{tid}.jpg"), img)
print(f"{len(test_cases)} test images (saved to {GEN_DIR.relative_to(BASE_DIR)})")

preview = [c for c in test_cases if c[2] == LANDMARKS[0]]
fig, axes = plt.subplots(1, len(preview), figsize=(4 * len(preview), 3.6))
for ax, (tid, cond, truth, img) in zip(axes, preview):
    ax.imshow(to_rgb(img))
    ax.set_title(f"{cond}\n{img.shape[1]} x {img.shape[0]} px")
    ax.axis("off")
fig.suptitle(f"Test conditions for {pretty(LANDMARKS[0])}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
results, rows = [], []
for tid, condition, truth, img in test_cases:
    r = recognize_landmark(img, name=tid, show=False, save=True)
    correct = (not r["recognized"]) if truth == "Unknown" else (r["recognized"] and r["prediction"] == truth)
    r.update(truth=truth, condition=condition, correct=correct)
    results.append(r)
    rows.append({
        "Test": tid, "Condition": condition, "Ground truth": pretty(truth),
        "Top candidate": pretty(r["top_candidate"]),
        "Predicted landmark": pretty(r["prediction"]) if r["recognized"] else "Unknown (rejected)",
        "Keypoints": r["n_keypoints"], "Reliable matches": r["n_good"], "RANSAC inliers": r["n_inliers"],
        "Margin": round(r["margin"], 2), "Decision": "Recognized" if r["recognized"] else "Rejected",
        "Correct": "Yes" if correct else "No", "Time (s)": round(r["time"], 3),
    })
    print(f"{tid:45s} {rows[-1]['Decision']:10s} {rows[-1]['Predicted landmark']:22s} {'ok' if correct else 'WRONG'}")

results_df = pd.DataFrame(rows)
results_df.to_csv(OUT_DIR / "experimental_results.csv", index=False)
print("\nTable of experimental results (saved to outputs/experimental_results.csv):")
display(results_df)

In [ ]:
summary = (results_df.groupby("Condition", sort=False)
           .agg(Tests=("Test", "count"),
                Correct=("Correct", lambda s: int((s == "Yes").sum())),
                Mean_keypoints=("Keypoints", "mean"),
                Mean_reliable_matches=("Reliable matches", "mean"),
                Mean_RANSAC_inliers=("RANSAC inliers", "mean"),
                Mean_time_s=("Time (s)", "mean"))
           .round(2))
summary.insert(2, "Accuracy (%)", (100 * summary["Correct"] / summary["Tests"]).round(1))
summary.to_csv(OUT_DIR / "summary_by_condition.csv")
display(summary)

n_ok = int((results_df["Correct"] == "Yes").sum())
print(f"Overall accuracy: {n_ok}/{len(results_df)} = {100 * n_ok / len(results_df):.1f} %")
print(f"Average processing time per query: {results_df['Time (s)'].mean():.3f} s "
      f"(min {results_df['Time (s)'].min():.3f} s, max {results_df['Time (s)'].max():.3f} s)")

### RANSAC inliers per landmark and condition

Darker = more geometrically verified matches. Cells below the `MIN_INLIERS` threshold are labelled *rejected*.

In [ ]:
known = results_df[results_df["Ground truth"] != "Unknown"]
cond_order = list(CONDITIONS.values())
inl = known.pivot(index="Ground truth", columns="Condition", values="RANSAC inliers")[cond_order]
dec = known.pivot(index="Ground truth", columns="Condition", values="Decision")[cond_order]

fig, ax = plt.subplots(figsize=(11, 4.6))
im = ax.imshow(inl.values, cmap=SEQ_BLUE, aspect="auto", vmin=0)
ax.set_xticks(range(len(cond_order)), cond_order)
ax.set_yticks(range(len(inl)), inl.index)
vmax = inl.values.max()
for i in range(inl.shape[0]):
    for j in range(inl.shape[1]):
        v = inl.values[i, j]
        label = f"{v}" if dec.values[i, j] == "Recognized" else f"{v}\nrejected"
        ax.text(j, i, label, ha="center", va="center", fontsize=11,
                color="white" if v > 0.55 * vmax else "#1f1f1f")
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.colorbar(im, ax=ax, label="RANSAC inliers")
ax.set_title(f"RANSAC inliers of the predicted landmark (acceptance threshold = {MIN_INLIERS})")
plt.tight_layout()
save_figure(fig, "task6_inlier_heatmap")
plt.show()

### Visual results of all tests (screenshots)

Green title = correct decision, red = wrong. Recognised landmarks show the localised region.

In [ ]:
def show_tile(ax, r, label):
    img = draw_localization(r["query"], r["loc"]) if (r["recognized"] and r["loc"]["valid"]) else r["query"]["image"]
    ax.imshow(to_rgb(img))
    ax.axis("off")
    verdict = pretty(r["prediction"]) if r["recognized"] else "Rejected"
    ax.set_title(f"{label}\n{verdict} - {r['n_good']} matches / {r['n_inliers']} inliers",
                 fontsize=9, color=OK_COLOR if r["correct"] else BAD_COLOR)


by_name = {r["name"]: r for r in results}
unknown_results = [r for r in results if r["truth"] == "Unknown"]
n_cols = len(CONDITIONS)
n_rows = len(LANDMARKS) + int(np.ceil(len(unknown_results) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for i, lm in enumerate(LANDMARKS):
    for j, (key, cond) in enumerate(CONDITIONS.items()):
        show_tile(axes[i, j], by_name[f"{lm}__{key}"], f"{pretty(lm)} - {cond}")
for k, r in enumerate(unknown_results):
    show_tile(axes[len(LANDMARKS) + k // n_cols, k % n_cols], r, f"Not in DB: {r['name'].split('__')[1]}")
fig.suptitle("Task 6 – recognition and localisation results for every test image", fontsize=15)
plt.tight_layout()
save_figure(fig, "task6_all_results")
plt.show()

### Detailed output for one example of each condition

In [ ]:
examples = ["Statue_of_Liberty__dark", "Eiffel_Tower__rotated", "Colosseum__resized",
            "Great_Wall_of_China__viewpoint", "Unknown__pyramids_of_giza"]
for name in examples:
    if name in by_name:
        plot_result(by_name[name])
        plt.show()

### Why CLAHE? – brightness test with and without contrast normalisation

The database is rebuilt without CLAHE and only the *Brightness 40%* tests are repeated. Without contrast normalisation the darkened images lose most of their keypoints, because SIFT's contrast threshold is absolute.

In [ ]:
ablation = []
USE_CLAHE = False
try:
    database = build_database(landmark_paths)
    for tid, condition, truth, img in test_cases:
        if condition != "Brightness 40%":
            continue
        r = recognize_landmark(img, name=tid, show=False, save=False)
        with_clahe = results_df.set_index("Test").loc[tid]
        ablation.append({
            "Test": tid,
            "Keypoints (CLAHE)": with_clahe["Keypoints"], "Keypoints (no CLAHE)": r["n_keypoints"],
            "Inliers (CLAHE)": with_clahe["RANSAC inliers"], "Inliers (no CLAHE)": r["n_inliers"],
            "Decision (CLAHE)": with_clahe["Decision"],
            "Decision (no CLAHE)": "Recognized" if r["recognized"] else "Rejected"})
finally:
    USE_CLAHE = True
    database = build_database(landmark_paths)   # restore the normal database
display(pd.DataFrame(ablation).set_index("Test"))

### Extra: challenging cases (beyond the required protocol)

Four harder photos test the limits of the system: an oblique view of the Taj Mahal through the garden trees, the Eiffel Tower photographed from directly underneath, a distant Eiffel Tower behind a crowd, and a *different section* of the Great Wall than the ones in the database. The ground truth is the part of the file name before `__`.

In [ ]:
challenging, ch_rows = [], []
for path in list_images(QUERY_DIR / "Challenging"):
    truth = path.stem.split("__")[0]
    r = recognize_landmark(load_image(path), name=f"challenging__{path.stem}", show=False, save=True)
    r.update(truth=truth, correct=r["recognized"] and r["prediction"] == truth)
    challenging.append(r)
    ch_rows.append({"Image": path.name, "Ground truth": pretty(truth), "Top candidate": pretty(r["top_candidate"]),
                    "Reliable matches": r["n_good"], "RANSAC inliers": r["n_inliers"], "Margin": round(r["margin"], 2),
                    "Decision": "Recognized" if r["recognized"] else "Rejected", "Time (s)": round(r["time"], 3)})
display(pd.DataFrame(ch_rows))

fig, axes = plt.subplots(1, len(challenging), figsize=(4.5 * len(challenging), 4), squeeze=False)
for ax, r in zip(axes[0], challenging):
    show_tile(ax, r, r["name"].split("__", 1)[1].replace("__", " - ").replace("_", " "))
plt.tight_layout()
save_figure(fig, "extra_challenging_cases")
plt.show()

### Download the outputs

Creates `lab06_outputs.zip` with all screenshots, the generated test images and the result tables (the large feature pickle is left out) and downloads it.

In [ ]:
archive = BASE_DIR / "lab06_outputs.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.rglob("*")):
        if f.is_file() and f.suffix != ".pkl":
            zf.write(f, f.relative_to(BASE_DIR))
print(f"{archive.name}: {archive.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("Not in Colab - the zip is next to the notebook.")

## 8. Discussion – performance and limitations

*(Values quoted here are approximate; check them against the tables produced above, since timings and exact counts vary slightly between machines and OpenCV versions.)*

### Performance

- **Recognition.** With the chosen thresholds, the known landmarks are recognised under every required condition and all five landmarks that are not in the database are rejected (see the summary table). Accepted landmarks have roughly 20 to 250+ RANSAC inliers, while the unknown landmarks stay at about 7 or fewer – a wide gap around the threshold of 15.
- **Rotation (45°).** SIFT assigns every keypoint a dominant orientation and builds its descriptor relative to it, so rotation hardly changes the result – the match count stays about the same (it can even increase, because the enlarged rotated canvas is larger than 800 px).
- **Scale (50%).** SIFT searches for keypoints over a Gaussian scale space, so features are found again at the smaller size. The image has only a quarter of the pixels, so keypoints drop to about a quarter and reliable matches to roughly a third to a half, but the inliers stay above the threshold. The Great Wall is the weakest case here because its landscape has few distinctive features.
- **Brightness (40%).** The descriptor is normalised and therefore robust to illumination, but the detector's fixed contrast threshold discards most keypoints of a dark image. The ablation shows that without CLAHE a darkened image loses most of its keypoints and low-texture landmarks can then be rejected; with CLAHE the keypoints, and with them the matches, are largely recovered.
- **Different viewpoint.** Photos taken from another position still match, with fewer inliers than the original view, as long as the change is moderate.
- **Rejecting unknown landmarks.** Photos of landmarks that are not in the database still get up to about 30 accidental reliable matches – sometimes *more* than a true but small/dark match. They are rejected because (a) these matches are spread evenly over all five landmarks (margin ≈ 1.0–1.5×) and (b) they do not agree on a common transformation (≤ about 7 RANSAC inliers). This is why the decision does not rely on the match count alone.
- **Speed.** A query takes roughly 0.2–6 s on a Colab CPU. Most of the time is spent brute-force matching thousands of query descriptors against all 25 reference images; images with many keypoints (rotated, detailed scenes) are the slowest.

### Limitations

1. **Large viewpoint changes / 3-D structure.** SIFT is invariant to rotation and scale but only partially to perspective. A homography models a *plane*, so it only approximates a 3-D building. Photos from very different angles (e.g. the Eiffel Tower from directly underneath, *Challenging* section) are rejected, while a moderate change such as the oblique Taj Mahal garden view is still recognised.
2. **Coverage of the database.** A landmark can only be recognised from viewpoints similar to one of its reference images. The Great Wall is thousands of kilometres long – a photo of a section that is not in the database cannot be matched. More reference images per landmark (different sides, seasons, day/night) would increase recall.
3. **Repetitive and low-texture content.** Bricks, windows, arches, sky, water and vegetation produce many similar descriptors; the ratio test removes them, which leaves landscapes and lattice structures with relatively few reliable matches.
4. **Illumination.** CLAHE handles global darkening, but strong non-linear changes (day vs. night lighting, harsh shadows, reflections) change the local gradients themselves and reduce matching.
5. **Scalability.** Brute-force matching grows linearly with the number of reference images. A real tourism system with thousands of landmarks would need approximate nearest-neighbour search (FLANN / KD-trees), a bag-of-visual-words or vocabulary-tree index, or compact global descriptors to shortlist candidates before geometric verification.
6. **Fixed thresholds.** `MIN_GOOD_MATCHES`, `MIN_MARGIN` and `MIN_INLIERS` were tuned on a small set of images; a larger validation set is needed to choose them reliably (precision/recall trade-off).
7. **Localisation meaning.** The red outline is the *reference photo's frame* projected into the query; it shows where the landmark is and how the views relate, but it is not a tight segmentation of the building.

### Possible improvements

RootSIFT descriptors, FLANN-based matching, voting over several reference images per landmark, fundamental-matrix verification for non-planar scenes, and learned features (e.g. DELF, SuperPoint + SuperGlue) for very large viewpoint changes.

### Conclusion

SIFT feature matching with Lowe's ratio test and RANSAC geometric verification is a practical basis for a tourism landmark recognition system: it identifies landmarks reliably under rotation, scale and (with contrast normalisation) brightness changes, localises them in the photo, and – thanks to the margin and inlier checks – rejects landmarks that it does not know instead of guessing.